# Diabetes model for TrustyAI SPD demo (AgeOver50)

This notebook rebuilds the diabetes demo model so that:

- the raw `Age` feature is replaced with a binary `AgeOver50`
- normalization is applied **inside the PyTorch model**
- the exported ONNX model accepts **raw feature values** at inference time
- `AgeOver50` remains a clean binary protected attribute for TrustyAI SPD

This avoids the common mismatch where a model is trained on scaled inputs but the deployed ONNX model receives raw inputs.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


In [2]:
DATA_PATH = Path("pima-indians-diabetes.data.csv")
MODEL_PATH = Path("diabetes.onnx")
TRAINING_DATA_EXPORT = Path("diabetes-trustyai-training-data.csv")
SAMPLE_PAYLOAD_DIR = Path("sample-payloads")

RANDOM_STATE = 42
TEST_SIZE = 0.20
EPOCHS = 300
LEARNING_RATE = 0.005

# Demo-only option: flip a fraction of favorable labels for AgeOver50 == 1
ENABLE_SYNTHETIC_BIAS = False
SYNTHETIC_BIAS_FLIP_RATE = 0.20

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## Load the dataset


In [3]:
names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv(DATA_PATH, names=names)
print(df.head())
print()
print(df.describe(include="all"))


   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002  

## Replace `Age` with `AgeOver50`


In [4]:
df["AgeOver50"] = (df["Age"] > 50).astype(np.float32)
df = df.drop(columns=["Age"])

feature_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "AgeOver50",
]
target_col = "Outcome"

continuous_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
]
binary_cols = ["AgeOver50"]

X_df = df[feature_cols].copy()
y = df[target_col].astype(np.int64).values

print("Feature columns:", feature_cols)
print()
print("AgeOver50 counts:")
print(df["AgeOver50"].value_counts().sort_index())


Feature columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'AgeOver50']

AgeOver50 counts:
AgeOver50
0.0    687
1.0     81
Name: count, dtype: int64


## Train/test split on **raw** inputs

We keep the dataframe values raw here.  
Normalization will be done inside the model, using statistics computed only from the training set.


In [5]:
X = X_df.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_df = pd.DataFrame(X_train, columns=feature_cols)
test_df = pd.DataFrame(X_test, columns=feature_cols)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


Training set shape: (614, 8)
Test set shape: (154, 8)


## Compute normalization statistics from the training set

Only the 7 continuous columns are normalized.  
`AgeOver50` stays as a literal `0.0/1.0`.


In [6]:
train_means = train_df[continuous_cols].mean().astype(np.float32)
train_stds = train_df[continuous_cols].std(ddof=0).replace(0, 1).astype(np.float32)

print("Training means:")
print(train_means)
print()
print("Training stds:")
print(train_stds)


Training means:
Pregnancies                   3.819218
Glucose                     120.908798
BloodPressure                69.442993
SkinThickness                20.776873
Insulin                      78.666122
BMI                          31.973289
DiabetesPedigreeFunction      0.477428
dtype: float32

Training stds:
Pregnancies                   3.311448
Glucose                      31.535381
BloodPressure                18.387589
SkinThickness                15.843515
Insulin                     107.648804
BMI                           7.854960
DiabetesPedigreeFunction      0.330031
dtype: float32


## Optional synthetic bias for a clearer SPD demo

Leave this disabled for a natural model.  
Enable it only if you want a stronger visible SPD signal in a lab/demo.


In [7]:
y_train_effective = y_train.copy()

if ENABLE_SYNTHETIC_BIAS:
    older_mask = train_df["AgeOver50"].values == 1.0
    favorable_mask = y_train_effective == 0

    candidate_idx = np.where(older_mask & favorable_mask)[0]
    rng = np.random.default_rng(RANDOM_STATE)
    flip_count = int(len(candidate_idx) * SYNTHETIC_BIAS_FLIP_RATE)

    if flip_count > 0:
        flip_idx = rng.choice(candidate_idx, size=flip_count, replace=False)
        y_train_effective[flip_idx] = 1

    print(f"Synthetic bias enabled. Flipped {flip_count} labels from 0 -> 1 for AgeOver50 == 1.")
else:
    print("Synthetic bias disabled.")


Synthetic bias disabled.


## Define the model with built-in normalization

This is the key fix.

The model accepts **raw feature values**.  
Inside `forward()`, it standardizes the continuous features using the stored training means/stds, leaves `AgeOver50` untouched, and then runs the dense network.

Because the normalization lives inside the PyTorch model, it is exported into ONNX too.


In [8]:
class DiabetesModel(nn.Module):
    def __init__(self, means, stds):
        super().__init__()

        self.register_buffer("means", torch.tensor(means.values, dtype=torch.float32))
        self.register_buffer("stds", torch.tensor(stds.values, dtype=torch.float32))

        self.net = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def preprocess(self, x):
        x_cont = x[:, :7]
        x_bin = x[:, 7:].clone()  # AgeOver50
        x_cont = (x_cont - self.means) / self.stds
        return torch.cat([x_cont, x_bin], dim=1)

    def forward(self, x):
        x = self.preprocess(x)
        logits = self.net(x)
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(probs, dim=1)
        return labels, probs


model = DiabetesModel(train_means, train_stds)
model


DiabetesModel(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=2, bias=True)
  )
)

## Train


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train_effective)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    logits = model.net(model.preprocess(X_train_t))
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:03d}/{EPOCHS} - loss: {loss.item():.4f}")


Epoch 001/300 - loss: 0.7585
Epoch 025/300 - loss: 0.5555
Epoch 050/300 - loss: 0.4591
Epoch 075/300 - loss: 0.4396
Epoch 100/300 - loss: 0.4231
Epoch 125/300 - loss: 0.3987
Epoch 150/300 - loss: 0.3703
Epoch 175/300 - loss: 0.3458
Epoch 200/300 - loss: 0.3229
Epoch 225/300 - loss: 0.3032
Epoch 250/300 - loss: 0.2895
Epoch 275/300 - loss: 0.2813
Epoch 300/300 - loss: 0.2748


## Evaluate on the test set


In [10]:
model.eval()

with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test)
    pred_labels, pred_probs = model(X_test_t)

y_pred = pred_labels.numpy()
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(float(accuracy), 4))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))


Accuracy: 0.7143

Confusion matrix:
[[77 23]
 [21 33]]

Classification report:
              precision    recall  f1-score   support

           0     0.7857    0.7700    0.7778       100
           1     0.5893    0.6111    0.6000        54

    accuracy                         0.7143       154
   macro avg     0.6875    0.6906    0.6889       154
weighted avg     0.7168    0.7143    0.7154       154



## Quick raw-input smoke test

These are raw feature values in the same order you will send to the deployed model.

If the model is healthy, these outputs should not all be identical.


In [11]:
raw_smoke = np.array([
    [0.0, 92.0, 68.0, 18.0, 80.0, 23.5, 0.18, 0.0],
    [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0],
    [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0],
    [5.0, 185.0, 96.0, 35.0, 240.0, 40.0, 0.82, 1.0],
], dtype=np.float32)

with torch.no_grad():
    smoke_labels, smoke_probs = model(torch.tensor(raw_smoke))

print("Labels:", smoke_labels.numpy().tolist())
print("Probabilities:")
print(smoke_probs.numpy())


Labels: [0, 1, 1, 1]
Probabilities:
[[9.9989617e-01 1.0386996e-04]
 [3.7650010e-01 6.2349993e-01]
 [5.3139590e-02 9.4686043e-01]
 [2.7819350e-04 9.9972183e-01]]


## Export ONNX

The exported model accepts **raw** inference payloads.


In [12]:
dummy_input = torch.randn(1, len(feature_cols), dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    MODEL_PATH.as_posix(),
    verbose=False,
    input_names=["dense_input"],
    output_names=["label", "probabilities"],
    dynamic_axes={
        "dense_input": {0: "batch_size"},
        "label": {0: "batch_size"},
        "probabilities": {0: "batch_size"},
    },
    opset_version=11,
)

print(f"Saved ONNX model to: {MODEL_PATH.resolve()}")


Saved ONNX model to: /opt/app-root/src/ai267-sa/ch05/diabetes-model/diabetes.onnx


## Export training data for TrustyAI

This export uses the raw feature values, including `AgeOver50`, which is what you want TrustyAI to see.


In [13]:
export_df = pd.DataFrame(X, columns=feature_cols)
export_df["Outcome"] = y

export_df.to_csv(TRAINING_DATA_EXPORT, index=False)
print(f"Saved training data export to: {TRAINING_DATA_EXPORT.resolve()}")
print(export_df.head())


Saved training data export to: /opt/app-root/src/ai267-sa/ch05/diabetes-model/diabetes-trustyai-training-data.csv
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin        BMI  \
0          6.0    148.0           72.0           35.0      0.0  33.599998   
1          1.0     85.0           66.0           29.0      0.0  26.600000   
2          8.0    183.0           64.0            0.0      0.0  23.299999   
3          1.0     89.0           66.0           23.0     94.0  28.100000   
4          0.0    137.0           40.0           35.0    168.0  43.099998   

   DiabetesPedigreeFunction  AgeOver50  Outcome  
0                     0.627        0.0        1  
1                     0.351        0.0        0  
2                     0.672        0.0        1  
3                     0.167        0.0        0  
4                     2.288        0.0        1  


## Save sample inference payloads


In [14]:
SAMPLE_PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

sample_under_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0],
        }
    ],
}

sample_over_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0],
        }
    ],
}

sample_batch = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [4, 8],
            "datatype": "FP32",
            "data": [
                0.0, 92.0, 68.0, 18.0, 80.0, 23.5, 0.18, 0.0,
                2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0,
                2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0,
                5.0, 185.0, 96.0, 35.0, 240.0, 40.0, 0.82, 1.0,
            ],
        }
    ],
}

for name, payload in {
    "under50.json": sample_under_50,
    "over50.json": sample_over_50,
    "batch.json": sample_batch,
}.items():
    p = SAMPLE_PAYLOAD_DIR / name
    p.write_text(json.dumps(payload, indent=2))
    print(f"Saved {p.resolve()}")


Saved /opt/app-root/src/ai267-sa/ch05/diabetes-model/sample-payloads/under50.json
Saved /opt/app-root/src/ai267-sa/ch05/diabetes-model/sample-payloads/over50.json
Saved /opt/app-root/src/ai267-sa/ch05/diabetes-model/sample-payloads/batch.json


## Example commands

Use these after redeploying the new ONNX model.


In [15]:
print(f'''
# Single infer request example
curl -sk \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  https://diabetes-myproj1.apps.ocp4.example.com/v2/models/diabetes/infer \
  -d '{json.dumps(sample_under_50)}' | jq

# TrustyAI info check
curl -sk -H "Authorization: Bearer $TOKEN" "$TRUSTY_ROUTE/info" | jq

# Example SPD request after new observations exist
curl -sk -H "Authorization: Bearer $TOKEN" \
  -H 'Content-Type: application/json' \
  -X POST "$TRUSTY_ROUTE/metrics/group/fairness/spd" \
  -d '{{
    "modelId": "diabetes",
    "protectedAttribute": "AgeOver50",
    "privilegedAttribute": {{"type":"FLOAT","value":0}},
    "unprivilegedAttribute": {{"type":"FLOAT","value":1}},
    "outcomeName": "Final_Prediction",
    "favorableOutcome": {{"type":"INT64","value":0}},
    "batchSize": 100
  }}' | jq
''')



# Single infer request example
curl -sk   -H "Authorization: Bearer $TOKEN"   -H "Content-Type: application/json"   https://diabetes-myproj1.apps.ocp4.example.com/v2/models/diabetes/infer   -d '{"model_name": "diabetes", "inputs": [{"name": "dense_input", "shape": [1, 8], "datatype": "FP32", "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0]}]}' | jq

# TrustyAI info check
curl -sk -H "Authorization: Bearer $TOKEN" "$TRUSTY_ROUTE/info" | jq

# Example SPD request after new observations exist
curl -sk -H "Authorization: Bearer $TOKEN"   -H 'Content-Type: application/json'   -X POST "$TRUSTY_ROUTE/metrics/group/fairness/spd"   -d '{
    "modelId": "diabetes",
    "protectedAttribute": "AgeOver50",
    "privilegedAttribute": {"type":"FLOAT","value":0},
    "unprivilegedAttribute": {"type":"FLOAT","value":1},
    "outcomeName": "Final_Prediction",
    "favorableOutcome": {"type":"INT64","value":0},
    "batchSize": 100
  }' | jq

